# 02 — Generate 3-Arm Grounding Artifacts (Stage 1 → 2 → 3)

Deteksi **YOLOv8 Stage 1** → bbox YANG SAMA feed ke 3 arm: **bbox / mask / hybrid** (mask dari SAM+adapter). Hanya spatial referent berubah → perbandingan terkontrol (draft §4.3).

**Prasyarat:** `yolov8_dentex.pt` (notebook 03) + `adapter_best.pth` (notebook 01) di Drive.

Eval set stratified (n per kelas) supaya biaya GPT-4o Stage 3 terkendali. Runtime: **GPU**.

## Cell 1 — Mount Drive + sync repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/opg-live'

import os
REPO = '/content/opg-live'
if not os.path.exists(REPO):
    !git clone https://github.com/AndreasTopuh/opg-live.git {REPO}
!cd {REPO} && git fetch origin && git reset --hard origin/main && git log --oneline -1

## Cell 2 — Install deps

In [ ]:
%pip install -q ultralytics segment-anything pycocotools opencv-python-headless
import torch; print('GPU:', torch.cuda.get_device_name(0))

## Cell 3 — Rebuild YOLO val split (held-out images)
Deteksi dijalankan di gambar val (yang TIDAK dilatih detektor). Split deterministik (seed sama dengan training).

In [ ]:
!python /content/opg-live/scripts/dentex_to_yolo.py --drive {DRIVE_ROOT} --out /content/yolo
!ls /content/yolo/images/val | head -3; echo '...'; ls /content/yolo/images/val | wc -l

## Cell 4 — Generate artifacts (mode YOLO)
Deteksi YOLO → SAM mask → render 3 arm. `--n_per_class 40` → ~160 deteksi × 3 arm. Manifest juga catat apakah diagnosis Stage 1 cocok GT.

In [ ]:
!python /content/opg-live/scripts/make_artifacts.py \
  --drive {DRIVE_ROOT} \
  --mode yolo \
  --yolo_ckpt {DRIVE_ROOT}/checkpoints/yolov8_dentex.pt \
  --images_dir /content/yolo/images/val \
  --sam_ckpt {DRIVE_ROOT}/checkpoints/sam_vit_h_4b8939.pth \
  --adapter {DRIVE_ROOT}/checkpoints/adapter_best.pth \
  --conf 0.25 --n_per_class 40

## Cell 5 — Preview: 1 deteksi, 3 arm berdampingan

In [ ]:
import json, matplotlib.pyplot as plt, cv2
ART = f'{DRIVE_ROOT}/outputs/artifacts'
man = [json.loads(l) for l in open(f'{ART}/manifest.jsonl')]
print('Total artifacts:', len(man))

m = man[0]
print('Stage 1 diagnosis:', m['pred_cls_name'], '| conf', m['conf'],
      '| correct vs GT:', m['correct'], '(IoU', m['gt_iou'], ')')
fig, ax = plt.subplots(1, 3, figsize=(20, 6))
for i, arm in enumerate(['bbox', 'mask', 'hybrid']):
    im = cv2.cvtColor(cv2.imread(f"{ART}/{m['artifacts'][arm]}"), cv2.COLOR_BGR2RGB)
    ax[i].imshow(im); ax[i].set_title(f"{arm} — {m['pred_cls_name']}"); ax[i].axis('off')
plt.tight_layout(); plt.show()

## ✅ Deliverable
- [ ] `outputs/artifacts/{bbox,mask,hybrid}/*.png` di Drive
- [ ] `manifest.jsonl` (det_id, pred_cls, conf, bbox, mask_area, matched_gt_cls, correct, 3 arm)
- [ ] preview 3 arm benar (box hijau / mask merah / hybrid)
- [ ] % diagnosis Stage 1 benar terlaporkan (detector quality)

**Next:** `04_explain_rag.ipynb` (Stage 3) — kirim tiap artifact ke GPT-4o (OpenRouter) + RAG → penjelasan L-F-V → metrik HR/GS/CTC per arm.